# Module 6: Regression with ARMA Errors

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Everything so far has modelled a series using only its own past. This module
adds **explanatory variables**: exposure, an external driver, a policy
indicator.

That last one is the reason this module exists. A policy indicator in a
regression is the simplest form of an intervention estimate, and it is the
direct precursor to [Module 11](Module_11_Interrupted_Time_Series.ipynb) and
to the whole causal inference series.

It is also where ordinary regression does its most expensive damage.

**About 30 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
final = monthly[monthly["provisional"] == 0]          # never fit on unfinished months


CALENDAR = pd.period_range("2019-01", "2026-04", freq="M").to_timestamp()


def counts(agency_id):
    """Monthly counts on a complete calendar, so a gap stays visible as missing."""
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    s = pd.Series(d["n_uof"].values, dtype=float,
                  index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    s = s.reindex(CALENDAR)
    s.index.freq = "MS"
    return s


def rate(agency_id):
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    s = pd.Series((100 * d["n_uof"] / d["n_arrests"]).values,
                  index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    s = s.reindex(CALENDAR)
    s.index.freq = "MS"
    return s


print(f"{final['agency_id'].nunique()} agencies, {final['year_month'].nunique()} months")

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import acf, pacf, adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox


def ljung(resid, lag=12):
    return float(acorr_ljungbox(resid, lags=[lag], return_df=True)["lb_pvalue"].iloc[0])

import statsmodels.formula.api as smf

g = (final[(final["agency_id"] == "A001") & (final["year_month"] <= "2025-12")]
     .sort_values("year_month"))                      # Stonewick, which adopted the training

y = pd.Series(np.log(g["n_uof"].values),
              index=pd.PeriodIndex(g["year_month"], freq="M").to_timestamp())
y.index.freq = "MS"

X = pd.DataFrame({
    "log_arrests": np.log(g["n_arrests"].values),
    "programme": (g["year_month"] >= "2023-11").astype(float).values,
}, index=y.index)

print(f"{len(y)} months, programme in place for {int(X['programme'].sum())} of them")

## 2. The model

> log(incidents) = intercept + b1 log(arrests) + b2 programme + **error**

and the error is allowed to be autocorrelated rather than assumed independent:

> error follows an ARMA process, seasonal terms included

The coefficients mean the same thing they would in ordinary regression.
**b2** is the proportional change associated with the programme. **b1** is the
elasticity with respect to arrests, and it should come out near 1: one percent
more arrests, one percent more incidents.

That second coefficient is a free sanity check, and most analyses throw it
away.

## 3. Fit it three ways

In [ ]:
ols = smf.ols("y ~ log_arrests + programme", data=X.assign(y=y.values)).fit()

seasonal_only = SARIMAX(y, exog=X, order=(0, 0, 0),
                        seasonal_order=(0, 1, 1, 12)).fit(disp=False)
arma = SARIMAX(y, exog=X, order=(0, 0, 1),
               seasonal_order=(0, 1, 1, 12)).fit(disp=False)

pct = lambda b: 100 * (np.exp(b) - 1)
rows = []
for name, mod in [("ordinary least squares", ols),
                  ("seasonal MA errors", seasonal_only),
                  ("MA(1) and seasonal MA errors", arma)]:
    lo, hi = mod.conf_int().loc["programme"]
    rows.append({"model": name,
                 "programme effect": f"{pct(mod.params['programme']):+.1f}%",
                 "95 percent interval": f"[{pct(lo):+.1f}, {pct(hi):+.1f}]",
                 "exposure coefficient": round(mod.params["log_arrests"], 2)})

pd.DataFrame(rows).set_index("model")

In [ ]:
print("the true programme effect built into the data: -12.0 percent")
print(f"\nOLS residuals, Ljung Box at lag 12:        p = {ljung(ols.resid):.4f}")
print(f"ARMA errors residuals, Ljung Box at lag 12: p = {ljung(arma.resid[13:]):.4f}")

## 4. What just happened

**Ordinary least squares gives −23.6 percent, about twice the truth, with a
narrow interval that excludes zero.** A confident, precise, wrong answer.

**Both ARMA error models give about −10 percent**, close to the true −12, with
an interval that includes zero.

Two separate things went wrong with OLS, and they are worth separating.

**The interval is wrong** because the errors are strongly autocorrelated. Its
Ljung Box p value is 0.0000. Intermediate
[Module 10](../../Intermediate/Module_10_Reading_Autocorrelation.md) explained
the mechanism; here it produces an interval roughly half the width it should
be.

**The point estimate is also wrong**, and that is the more serious failure.
OLS has no seasonal term, so the seasonal pattern has nowhere to go but into
the regressors, and the programme indicator absorbs part of it.

Look at the exposure coefficient for the diagnosis. Theory says it should be
near 1. **OLS returns 1.51**, which would mean a one percent rise in arrests
brings a one and a half percent rise in incidents. The ARMA error models return
0.89 and 0.93.

**A coefficient you can check against theory is the cheapest diagnostic in the
model, and it flagged this before any residual plot was drawn.**

## 5. Exposure as a regressor or as an offset

[Module 2](Module_02_Counts_Are_Not_Gaussian.ipynb) put exposure in as an
**offset**, fixing its coefficient at 1. Here it is a free **regressor**. Both
are defensible and they answer slightly different questions.

In [ ]:
free = arma.params["log_arrests"]
lo, hi = arma.conf_int().loc["log_arrests"]
print(f"  estimated exposure coefficient: {free:.2f}   95 percent interval "
      f"[{lo:.2f}, {hi:.2f}]")
print(f"  does the interval contain 1?  {bool(lo <= 1 <= hi)}")

The interval contains 1, so fixing it at 1 costs nothing here and buys a
degree of freedom.

**The rule: leave it free first and look.** If the interval contains 1, fix it
as an offset and say you checked. If it does not, the exposure is not behaving
proportionally and something about the denominator needs explaining before any
rate is reported at all.

## 6. This is not yet a causal estimate

The programme coefficient is now defensible as a **description**: over this
window, at this agency, incidents ran about 10 percent below what arrests and
the seasonal pattern would predict, after the programme began.

It is not an effect. The model has no comparison group, so anything else that
changed at Stonewick in November 2023 is inside that coefficient. Intermediate
[Module 16](../../Intermediate/Module_16_Did_Something_Change.md) made the
same point without the machinery; the machinery does not repair it.

What the machinery does add is an honest interval. **The interval here includes
zero**, which is the correct conclusion from one agency and matches what
Intermediate Module 16 found: a 12 percent effect is not measurable from a
single department.

## 7. What to carry away

| Habit | Why |
|---|---|
| Model the error structure, do not assume it away | the interval is otherwise meaningless |
| Include the seasonal term even with regressors | otherwise the season leaks into the coefficients |
| Check any coefficient with a known expected value | the cheapest diagnostic available |
| Report the interval, not only the coefficient | a single agency rarely supports a firm number |
| Do not call a policy coefficient an effect | there is no comparison group in this model |

## Exercise

Fit the same three models to Tarnbridge, which also adopted the programme. Does
OLS overstate the effect there too?

In [ ]:
# Fill in the blank, then run.
AGENCY = None              # try "A002"

if AGENCY:
    g2 = (final[(final["agency_id"] == AGENCY) & (final["year_month"] <= "2025-12")]
          .sort_values("year_month"))
    g2 = g2[g2["year_month"] != "2021-06"] if AGENCY == "A002" else g2
    y2 = pd.Series(np.log(g2["n_uof"].values),
                   index=pd.PeriodIndex(g2["year_month"], freq="M").to_timestamp())
    X2 = pd.DataFrame({"log_arrests": np.log(g2["n_arrests"].values),
                       "programme": (g2["year_month"] >= "2023-11").astype(float).values},
                      index=y2.index)
    o2 = smf.ols("y ~ log_arrests + programme", data=X2.assign(y=y2.values)).fit()
    a2 = SARIMAX(y2.reset_index(drop=True), exog=X2.reset_index(drop=True),
                 order=(0, 0, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
    for nm, mod in [("ordinary least squares", o2), ("ARMA errors", a2)]:
        lo, hi = mod.conf_int().loc["programme"]
        print(f"  {nm:24s} {pct(mod.params['programme']):+6.1f}%  "
              f"[{pct(lo):+6.1f}, {pct(hi):+6.1f}]   exposure "
              f"{mod.params['log_arrests']:+.2f}")
    print("  the truth: -12.0 percent")
else:
    print("Set AGENCY above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
AGENCY = "A002"
```

Compare the two exposure coefficients before anything else. Whichever model
returns something far from 1 is the one mis attributing variation, and it will
usually be the one without a seasonal term.

Two cautions specific to Tarnbridge. Its June 2021 unrest month is excluded
above, and leaving it in moves the estimate; that decision belongs in the
methods note, as Beginner
[Topic 11](../../Beginner/Topic_11_Outliers_And_Spikes.md) argued. And
Tarnbridge is smaller than Stonewick, so its interval will be wider still.
Neither agency on its own can settle whether the programme worked, which is
why [Module 10](Module_10_Panel_And_Hierarchical.ipynb) fits them together.

</details>

---

**Next:** [Module 7, State Space, Unobserved Components and ETS](Module_07_State_Space_And_ETS.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*